# Shared preprocessing 
This notebook builds the cleaned listening data that all research questions start from. 

Run it to get a local copy, as the output is gitignored (to big file for git).

**Input:** Currently the 2023 part of the ListenBrainz extract: `Extract/extract/year=2023/part-*.parquet`. How the extract was made and checked is described in `Claude Reports/04_Extract`.

**Output** (in `Data/processed data/`):
* `users_2023.csv`: the users we work with (`user_id` and `n_listens`)
* `listens_2023.parquet`: every cleaned listen of those users, one row per listen, ordered by user and time 

**Steps**
1. Pick the users
2. Clean their listens
3. Look at the result

**Requirements:** the `damml2026` env with `requirements.txt` installed (needs `duckdb` and `pyarrow`).

In [ ]:
import sys
from pathlib import Path
import duckdb
import pandas as pd

sys.path.append("../..")  # make the Assignment 02 folder (and src/) importable
from src.pick_users import pick_users
from src.clean_2a_exact_repeats import drop_duplicate_listens
from src.clean_2b_missing_names import drop_missing_names
from src.clean_2c_case_repeats import drop_case_repeats
from src.clean_2d_bulk_clusters import drop_bulk_clusters
from src.save_listens import save_listens

EXTRACT_PATH = Path("../../Extract/extract")        # input: year=YYYY/part-*.parquet (gitignored)
PROCESSED_DIR = Path("../../Data/processed data")   # all generated data goes here (gitignored)

USERS_PATH = PROCESSED_DIR / "users_2023.csv"          # output of step 1
LISTENS_PATH = PROCESSED_DIR / "listens_2023.parquet"  # output of step 2

if not any((EXTRACT_PATH / "year=2023").glob("part-*.parquet")):
    raise FileNotFoundError(f"No extract found in {EXTRACT_PATH.resolve()}/year=2023 - see Claude Reports/04_Extract")

## Step 1: pick the users

We use the same two-part rule as the group's 600-user test sample (`Samples/01_Test_Sample (600 users)`):

1. **Pool** Users with **2,000 to 40,000 listens in 2023**. This gives 10,558 users.
2. **Pick:** from the pool, the `N_USERS` users with the smallest `hash(user_id, seed)`. This is a random pick that gives the same users every time the same seed is used.

**Getting the same users as the sample.** 

The sample was made with exactly this rule: `N_USERS = 600` and `SEED = 2023`, so those settings give the exact same 600 users as in the sample.

* Unlike the sample, the next step keeps all their listens: the sample kept only ~3% of each user's listens at random, which is fine for counts and genres but breaks up listening sequences.*

* `N_USERS = None` keeps the whole pool (10,558 users, ~118 M listens); 
* another `SEED` would give a different random group.

The pick is saved to `users_2023.csv` as a record, if a later run picks different users than the ones already in the file, the function prints a warning.

In [ ]:
MIN_LISTENS = 2000   # at least this many listens in 2023 (exact repeats not counted)
MAX_LISTENS = 40000  # at most this many
N_USERS = 600        # None = the whole pool
SEED = 2023          # same seed as the sample

users = pick_users(EXTRACT_PATH, MIN_LISTENS, MAX_LISTENS, N_USERS, SEED, USERS_PATH)
users.describe()

## Step 2: clean the listens

A listens of the picked users are read from the extract and cleaned in four steps, each with its own script in `src/`: 

| step | script | what it removes |
|---|---|---|
| 2A | `clean_2a_exact_repeats.py` | drops exact repeats (same user, second, artist and track)|
| 2B | `clean_2b_missing_names.py` | listens without an artist or track name |
| 2C | `clean_2c_case_repeats.py` | repeats that only differ in upper/lower case |
| 2D | `clean_2d_bulk_clusters.py` | bulk-import clusters (many listens in one second) |

Must be run in sequence: 2A, 2B, 2C, 2D. (rerun, also needs preceeding steps)

**Not done: dropping short plays.** We could also have dropped listens shorter than 30 s (skips), but decided not to, as only ~32% of the listens record how long they were played (`ms_played`). Of the listens that do have this, only 0.5% are under 30 s.

In [ ]:
con = duckdb.connect()  # holds the 'listens' table through steps 2A-2D

### 2A: load the listens and drop duplicates

Reads every listen of the users in in the `USERS_PATH` from the extract, keeping the columns that are useful later. 

Then drops the duplicate listens: the same user, second, artist and track more than once. (These are double submissions, e.g. when a listen was both scrobbled live and imported later).

Columns kept: names, MusicBrainz ids, Spotify id, ISRC, durations, source

In [ ]:
drop_duplicate_listens(con, EXTRACT_PATH, USERS_PATH);

### 2B: drop listens without an artist or track name

A listen without an artist or track name cannot be matched to a song, so it is of no use for any research question.

In [ ]:
drop_missing_names(con);

### 2C: drop repeats that only differ in upper/lower case

`is_dup` in 2A compares names exactly, so `Daft Punk - One More Time` and `daft punk - one more time` at the same second by the same user both survive it. They are one listen, sent twice by different apps. The copy with a `recording_mbid` is kept when there is one.

In [ ]:
drop_case_repeats(con);

### 2D: drop bulk-import clusters

Some importers give a whole batch of listens the same timestamp. For the 600 users, ~96% of listens are alone in their second and ~4% come in pairs (likely real: timestamps are rounded to the second), but a few hundred seconds hold 6 to 56 listens, mostly from the *ListenBrainz Archive Importer*. No one plays that many tracks in one second, and they would create fake sessions with zero gaps. All listens of a second with more than `MAX_SAME_SECOND` listens are dropped.

In [ ]:
MAX_SAME_SECOND = 5  # more listens than this in one second (same user) = bulk-import cluster

drop_bulk_clusters(con, MAX_SAME_SECOND);

### Save the cleaned listens

Writes `listens_2023.parquet`: one row per listen, ordered by user and time, with `listen_time_utc` added next to `ts` (Unix seconds, UTC). It is saved as **parquet** rather than csv: much smaller, much faster to read, and it keeps the column types.

In [ ]:
save_listens(con, LISTENS_PATH)
con.close()

## Step 3: look at the result

`listens_2023.parquet` is the starting point for the RQ notebooks. For 600 users it fits in pandas (`pd.read_parquet`); for the whole pool, query it with DuckDB instead.

In [ ]:
listens = pd.read_parquet(LISTENS_PATH)
print(f"{len(listens):,} listens, {listens['user_id'].nunique():,} users, "
      f"{listens['listen_time_utc'].min():%Y-%m-%d} to {listens['listen_time_utc'].max():%Y-%m-%d}")
listens.head()

In [ ]:
# how complete is each column? (share of listens with a value)
listens.notna().mean().round(3).sort_values(ascending=False).to_frame("share_filled")

Only ~1% of listens carry a `recording_mbid`, so most listens are known only by their artist and track **names**. The next preprocessing step (not coded yet) matches these names to MusicBrainz ids with the canonical dump (`Raw data/MusicBrainz Canonical`), which then links each listen to genre and artist information (`Claude Reports/06_Artist_Table`).